# Problem Statement

Analyze GeoInsights 2021–2024 sales data to identify key revenue and profitability drivers across products, customers, channels, and U.S. regions; evaluate sales trends and transaction-level pricing and margin patterns; and compare 2024 product performance against budget targets. Use these insights to support data-driven product, channel, and regional growth decisions.

**Objective**

Deliver actionable insights from GeoInsights 2021–2024 sales data to:

1. Identify top-performing products, customers, channels, and regions by revenue and profitability.
2. Analyze monthly and yearly sales trends and investigate unusual patterns.
3. Evaluate transaction-level pricing, costs, and profit margins.
4. Compare 2024 product revenue against budget targets.
5. Support data-driven product, channel, and regional growth decisions.

These findings will guide the development of an interactive Power BI dashboard to support data-driven decision-making and sustainable business growth.

# **🔄 Data Ingestion**

In [63]:
import pandas as pd 
import numpy as np

In [64]:
workbook = pd.read_excel(
    'Regional Sales Dataset.xlsx',
    sheet_name=None
)

In [65]:
print("Sheets loaded successfully:")
print(list(workbook.keys()))

Sheets loaded successfully:
['Sales Orders', 'Customers', 'Regions', 'State Regions', 'Products', '2024 Budgets']


In [66]:
# Assign to named DataFrames
df_sales       = workbook ['Sales Orders']
df_customers   = workbook ['Customers']
df_products    = workbook ['Products']
df_regions     = workbook ['Regions']
df_state_reg   = workbook ['State Regions']
df_budgets     = workbook ['2024 Budgets']

# **🔍  Initial Inspection**

In [67]:
#   QUICK SHAPE OVERVIEW
print(f"df_sales      shape: {df_sales.shape}      # Sales Orders")
print(f"df_customers  shape: {df_customers.shape}  # Customers")
print(f"df_products   shape: {df_products.shape}   # Products")
print(f"df_regions    shape: {df_regions.shape}    # Regions")
print(f"df_state_reg  shape: {df_state_reg.shape}  # State Regions")
print(f"df_budgets    shape: {df_budgets.shape}    # 2024 Budgets")

df_sales      shape: (64104, 12)      # Sales Orders
df_customers  shape: (175, 2)  # Customers
df_products   shape: (30, 2)   # Products
df_regions    shape: (994, 15)    # Regions
df_state_reg  shape: (48, 3)  # State Regions
df_budgets    shape: (30, 2)    # 2024 Budgets


In [68]:
# Quick view of all raw DataFrames
dataframes = {
    'Sales Orders': df_sales,
    'Customers': df_customers,
    'Products': df_products,
    'Regions': df_regions,
    'State Regions': df_state_reg,
    '2024 Budgets': df_budgets
}

for name, dataframe in dataframes.items():
    print(f"\n{name} — Sample Records")
    display(dataframe.head())


Sales Orders — Sample Records


,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost
0,SO - 000225,2021-01-01 00:00:00,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343
1,SO - 0003378,2021-01-01 00:00:00,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918
2,SO - 0005126,2021-01-01 00:00:00,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740
3,SO - 0005614,2021-01-01 00:00:00,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852
4,SO - 0005781,2021-01-01 00:00:00,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270



Customers — Sample Records


,Customer Index,Customer Names
0,1,Geiss Company
1,2,Jaxbean Group
2,3,Ascend Ltd
3,4,Eire Corp
4,5,Blogtags Ltd



Products — Sample Records


...

,id,name,county,state_code,state,type,latitude,longitude,area_code,population,households,median_income,land_area,water_area,time_zone
0,1,Auburn,Lee County,AL,Alabama,City,32.60986,-85.48078,334,62059,21767,38342,152375113,2646161,America/Chicago
1,2,Birmingham,Shelby County/Jefferson County,AL,Alabama,City,33.52744,-86.79905,205,212461,89972,31061,378353942,6591013,America/Chicago
2,3,Decatur,Limestone County/Morgan County,AL,Alabama,City,34.57332,-86.99214,256,55437,22294,41496,141006257,17594716,America/Chicago
3,4,Dothan,Dale County/Houston County/Henry County,AL,Alabama,City,31.23370,-85.40682,334,68567,25913,42426,232166237,835468,America/Chicago
4,5,Hoover,Shelby County/Jefferson County,AL,Alabama,City,33.37695,-86.80558,205,84848,32789,77146,122016784,2553332,America/Chicago



State Regions — Sample Records


,State Code,State,Region
0,AL,Alabama,South
1,AR,Arkansas,South
2,AZ,Arizona,West
3,CA,California,West
4,CO,Colorado,West



2024 Budgets — Sample Records


,Product Name,2024 Budgets
0,Product 1,3016489.209
1,Product 2,3050087.565
2,Product 3,2642352.432
3,Product 4,2885560.824
4,Product 5,3925424.542


#  DATA OVERVIEW

In [69]:
# Check missing values across all source tables

for name, dataframe in dataframes.items():
    total_missing = dataframe.isna().sum().sum()

    print(f"{name}: {total_missing} missing values")

Sales Orders: 0 missing values
Customers: 0 missing values
Products: 0 missing values
Regions: 0 missing values
State Regions: 0 missing values
2024 Budgets: 0 missing values


In [70]:
# Validate OrderDate values

parsed_dates = pd.to_datetime(
    df_sales['OrderDate'],
    errors='coerce'
)

invalid_date_count = parsed_dates.isna().sum()

print(
    "Invalid OrderDate values:",
    invalid_date_count
)

Invalid OrderDate values: 36


In [71]:
df_sales.loc[
    parsed_dates.isna(),
    ['OrderNumber', 'OrderDate']
].head()

,OrderNumber,OrderDate
33393,SO - 0002023,29-02-2023
33394,SO - 0004899,29-02-2023
33395,SO - 0006305,29-02-2023
33396,SO - 0006626,29-02-2023
33397,SO - 000702,29-02-2023


In [72]:
for name, dataframe in dataframes.items():
    print(
        f"{name}: {dataframe.duplicated().sum()} duplicate rows"
    )

Sales Orders: 0 duplicate rows
Customers: 0 duplicate rows
Products: 0 duplicate rows
Regions: 0 duplicate rows
State Regions: 0 duplicate rows
2024 Budgets: 0 duplicate rows


# 🧹 Data Cleaning & Wrangling

In [73]:
df = df_sales.merge(
    df_customers,
    how='left',
    left_on='Customer Name Index',
    right_on='Customer Index',
    validate='many_to_one'
)

In [74]:
print("Rows after customer merge:", len(df))

Rows after customer merge: 64104


In [75]:
df.head(2)

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost,Customer Index,Customer Names
0,SO - 000225,2021-01-01 00:00:00,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343,126,Rhynoodle Ltd
1,SO - 0003378,2021-01-01 00:00:00,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918,96,Thoughtmix Ltd


In [76]:
# Merge with Products

df = df.merge(
    df_products,
    how='left',
    left_on='Product Description Index',
    right_on='Index',
    validate='many_to_one'
)

In [77]:
print("Rows after product merge:", len(df))
print(
    "Unmatched products:",
    df['Product Name'].isna().sum()
)

Rows after product merge: 64104
Unmatched products: 0


In [78]:
# Merge with Regions

df = df.merge(
    df_regions,
    how='left',
    left_on='Delivery Region Index',
    right_on='id',
    validate='many_to_one'
)

In [79]:
print("Rows after region merge:", len(df))

print(
    "Unmatched regions:",
    df['name'].isna().sum()
)

Rows after region merge: 64104
Unmatched regions: 0


In [80]:
# Merge State with US Region

df = df.merge(
    df_state_reg[['State Code', 'Region']],
    how='left',
    left_on='state_code',
    right_on='State Code',
    validate='many_to_one'
)

In [81]:
print("Rows after state-region merge:", len(df))

print(
    "Unmatched US regions:",
    df['Region'].isna().sum()
)

Rows after state-region merge: 64104
Unmatched US regions: 0


In [82]:
# Prepare 2024 Product Budget as a separate table

budget_df = df_budgets.rename(columns={
    'Product Name': 'product_name',
    '2024 Budgets': 'budget'
}).copy()

budget_df['year'] = 2024

budget_df.head()

,product_name,budget,year
0,Product 1,3016489.209,2024
1,Product 2,3050087.565,2024
2,Product 3,2642352.432,2024
3,Product 4,2885560.824,2024
4,Product 5,3925424.542,2024


In [83]:
print("Budget rows:", len(budget_df))
print(
    "Unique budget products:",
    budget_df['product_name'].nunique()
)
print(
    "Total 2024 Budget:",
    budget_df['budget'].sum()
)

Budget rows: 30
Unique budget products: 30
Total 2024 Budget: 62700262.337


In [84]:
# Display all columns for inspection
pd.set_option('display.max_columns', None)

display(df.head())

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost,Customer Index,Customer Names,Index,Product Name,id,name,county,state_code,state,type,latitude,longitude,area_code,population,households,median_income,land_area,water_area,time_zone,State Code,Region
0,SO - 000225,2021-01-01 00:00:00,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343,126,Rhynoodle Ltd,27,Product 27,364,Savannah,Chatham County,GA,Georgia,City,32.08354,-81.09983,912,145674,52798,36466,268318796,13908113,America/New York,GA,South
1,SO - 0003378,2021-01-01 00:00:00,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918,96,Thoughtmix Ltd,20,Product 20,488,Greenwood,Johnson County,IN,Indiana,City,39.61366,-86.10665,317,55586,20975,54176,72276415,1883,America/Indiana/Indianapolis,IN,Midwest
2,SO - 0005126,2021-01-01 00:00:00,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740,8,Amerisourc Corp,26,Product 26,155,Pleasanton,Alameda County,CA,California,City,37.66243,-121.87468,925,79510,26020,124759,62489257,386195,America/Los Angeles,CA,West
3,SO - 0005614,2021-01-01 00:00:00,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852,42,Colgate-Pa Group,7,Product 7,473,Bloomington,Monroe County,IN,Indiana,City,39.16533,-86.52639,812,84067,30232,30019,60221613,475857,America/Indiana/Indianapolis,IN,Midwest
4,SO - 0005781,2021-01-01 00:00:00,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270,73,Deseret Group,8,Product 8,256,Manchester,Hartford County,CT,Connecticut,Town,41.77524,-72.52443,959,58007,24141,63158,70972793,720300,America/New York,CT,Northeast


In [85]:
# Remove redundant columns created during merges

cols_to_drop = [
    'Customer Index',
    'Index',
    'id',
    'State Code'
]

df = df.drop(columns=cols_to_drop)

print("Shape after removing redundant columns:", df.shape)

Shape after removing redundant columns: (64104, 29)


In [86]:
display(df.head(2))

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost,Customer Names,Product Name,name,county,state_code,state,type,latitude,longitude,area_code,population,households,median_income,land_area,water_area,time_zone,Region
0,SO - 000225,2021-01-01 00:00:00,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343,Rhynoodle Ltd,Product 27,Savannah,Chatham County,GA,Georgia,City,32.08354,-81.09983,912,145674,52798,36466,268318796,13908113,America/New York,South
1,SO - 0003378,2021-01-01 00:00:00,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918,Thoughtmix Ltd,Product 20,Greenwood,Johnson County,IN,Indiana,City,39.61366,-86.10665,317,55586,20975,54176,72276415,1883,America/Indiana/Indianapolis,Midwest


In [87]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower()

# Verify
print(df.columns.tolist())

['ordernumber', 'orderdate', 'customer name index', 'channel', 'currency code', 'warehouse code', 'delivery region index', 'product description index', 'order quantity', 'unit price', 'line total', 'total unit cost', 'customer names', 'product name', 'name', 'county', 'state_code', 'state', 'type', 'latitude', 'longitude', 'area_code', 'population', 'households', 'median_income', 'land_area', 'water_area', 'time_zone', 'region']


# Selecting Relevant Columns for Analysis

In [88]:
# Select columns required for analysis
cols_to_keep = [
    'ordernumber',          # Order reference (not unique)
    'orderdate',            # Transaction date
    'customer names',       # Customer name
    'channel',              # Sales channel
    'product name',         # Product purchased
    'order quantity',       # Units ordered
    'unit price',           # Price per unit
    'line total',           # Revenue
    'total unit cost',      # Cost per unit
    'name',                 # City
    'state_code',           # State code
    'state',                # State name
    'region',               # U.S. region
    'latitude',             # Geographic latitude
    'longitude'             # Geographic longitude
]

df = df[cols_to_keep].copy()

print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (64104, 15)
['ordernumber', 'orderdate', 'customer names', 'channel', 'product name', 'order quantity', 'unit price', 'line total', 'total unit cost', 'name', 'state_code', 'state', 'region', 'latitude', 'longitude']


In [89]:
# Rename columns for clarity and consistency
df = df.rename(columns={
    'ordernumber'      : 'order_number',
    'orderdate'        : 'order_date',
    'customer names'   : 'customer_name',
    'channel'          : 'channel',
    'product name'     : 'product_name',
    'order quantity'   : 'quantity',
    'unit price'       : 'unit_price',
    'line total'       : 'revenue',
    'total unit cost'  : 'unit_cost',
    'name'             : 'city',
    'state_code'       : 'state',
    'state'            : 'state_name',
    'region'           : 'us_region',
    'latitude'         : 'lat',
    'longitude'        : 'lon'
})

# Create a unique transaction identifier
df.insert(0, 'transaction_id', range(1, len(df) + 1))

print(df.columns.tolist())
df.head()

['transaction_id', 'order_number', 'order_date', 'customer_name', 'channel', 'product_name', 'quantity', 'unit_price', 'revenue', 'unit_cost', 'city', 'state', 'state_name', 'us_region', 'lat', 'lon']


,transaction_id,order_number,order_date,customer_name,channel,product_name,quantity,unit_price,revenue,unit_cost,city,state,state_name,us_region,lat,lon
0,1,SO - 000225,2021-01-01 00:00:00,Rhynoodle Ltd,Wholesale,Product 27,6,2499.1,14994.6,1824.343,Savannah,GA,Georgia,South,32.08354,-81.09983
1,2,SO - 0003378,2021-01-01 00:00:00,Thoughtmix Ltd,Distributor,Product 20,11,2351.7,25868.7,1269.918,Greenwood,IN,Indiana,Midwest,39.61366,-86.10665
2,3,SO - 0005126,2021-01-01 00:00:00,Amerisourc Corp,Wholesale,Product 26,6,978.2,5869.2,684.740,Pleasanton,CA,California,West,37.66243,-121.87468
3,4,SO - 0005614,2021-01-01 00:00:00,Colgate-Pa Group,Export,Product 7,7,2338.3,16368.1,1028.852,Bloomington,IN,Indiana,Midwest,39.16533,-86.52639
4,5,SO - 0005781,2021-01-01 00:00:00,Deseret Group,Wholesale,Product 8,8,2291.4,18331.2,1260.270,Manchester,CT,Connecticut,Northeast,41.77524,-72.52443


In [90]:
# Identify invalid date: 29-02-2023
invalid_date_mask = (
    df['order_date']
    .astype(str)
    .str.strip()
    .eq('29-02-2023')
)

print("Invalid dates found:", invalid_date_mask.sum())

# 2023 is not a leap year.
# Assumption: these records represent the end of February.
df.loc[invalid_date_mask, 'order_date'] = pd.Timestamp('2023-02-28')

# Convert to datetime
df['order_date'] = pd.to_datetime(
    df['order_date'],
    errors='raise'
)

print("Missing dates:", df['order_date'].isna().sum())
print("Date range:", df['order_date'].min(), "to", df['order_date'].max())

Invalid dates found: 36
Missing dates: 0
Date range: 2021-01-01 00:00:00 to 2025-02-28 00:00:00


In [91]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 64104 entries, 0 to 64103
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   transaction_id  64104 non-null  int64         
 1   order_number    64104 non-null  str           
 2   order_date      64104 non-null  datetime64[us]
 3   customer_name   64104 non-null  str           
 4   channel         64104 non-null  str           
 5   product_name    64104 non-null  str           
 6   quantity        64104 non-null  int64         
 7   unit_price      64104 non-null  float64       
 8   revenue         64104 non-null  float64       
 9   unit_cost       64104 non-null  float64       
 10  city            64104 non-null  str           
 11  state           64104 non-null  str           
 12  state_name      64104 non-null  str           
 13  us_region       64104 non-null  str           
 14  lat             64104 non-null  float64       
 15  lon          

In [92]:
# Final missing-value validation
missing_values = df.isna().sum()

print(missing_values)
print("\nTotal missing values:", missing_values.sum())

transaction_id    0
order_number      0
order_date        0
customer_name     0
channel           0
product_name      0
quantity          0
unit_price        0
revenue           0
unit_cost         0
city              0
state             0
state_name        0
us_region         0
lat               0
lon               0
dtype: int64

Total missing values: 0


## 🛠️ Feature Engineering

Create analytical features for time-based, revenue, cost, and profitability analysis.

In [94]:
df.head()

,transaction_id,order_number,order_date,customer_name,channel,product_name,quantity,unit_price,revenue,unit_cost,city,state,state_name,us_region,lat,lon
0,1,SO - 000225,2021-01-01,Rhynoodle Ltd,Wholesale,Product 27,6,2499.1,14994.6,1824.343,Savannah,GA,Georgia,South,32.08354,-81.09983
1,2,SO - 0003378,2021-01-01,Thoughtmix Ltd,Distributor,Product 20,11,2351.7,25868.7,1269.918,Greenwood,IN,Indiana,Midwest,39.61366,-86.10665
2,3,SO - 0005126,2021-01-01,Amerisourc Corp,Wholesale,Product 26,6,978.2,5869.2,684.740,Pleasanton,CA,California,West,37.66243,-121.87468
3,4,SO - 0005614,2021-01-01,Colgate-Pa Group,Export,Product 7,7,2338.3,16368.1,1028.852,Bloomington,IN,Indiana,Midwest,39.16533,-86.52639
4,5,SO - 0005781,2021-01-01,Deseret Group,Wholesale,Product 8,8,2291.4,18331.2,1260.270,Manchester,CT,Connecticut,Northeast,41.77524,-72.52443


In [95]:
df['order_year'] = df['order_date'].dt.year

df['order_quarter'] = 'Q' + df['order_date'].dt.quarter.astype(str)

df['order_month_name'] = df['order_date'].dt.month_name()

df['order_month_num'] = df['order_date'].dt.month

In [97]:
# Calculate total transaction cost
df['total_cost'] = df['quantity'] * df['unit_cost']

# Calculate profit
df['profit'] = df['revenue'] - df['total_cost']

# Calculate profit margin percentage
df['profit_margin_percentage'] = np.where(
    df['revenue'] != 0,
    (df['profit'] / df['revenue']) * 100,
    np.nan
)

In [98]:
df.head()

,transaction_id,order_number,order_date,customer_name,channel,product_name,quantity,unit_price,revenue,unit_cost,city,state,state_name,us_region,lat,lon,order_year,order_quarter,order_month_name,order_month_num,total_cost,profit,profit_margin_percentage
0,1,SO - 000225,2021-01-01,Rhynoodle Ltd,Wholesale,Product 27,6,2499.1,14994.6,1824.343,Savannah,GA,Georgia,South,32.08354,-81.09983,2021,Q1,January,1,10946.058,4048.542,27.0
1,2,SO - 0003378,2021-01-01,Thoughtmix Ltd,Distributor,Product 20,11,2351.7,25868.7,1269.918,Greenwood,IN,Indiana,Midwest,39.61366,-86.10665,2021,Q1,January,1,13969.098,11899.602,46.0
2,3,SO - 0005126,2021-01-01,Amerisourc Corp,Wholesale,Product 26,6,978.2,5869.2,684.740,Pleasanton,CA,California,West,37.66243,-121.87468,2021,Q1,January,1,4108.440,1760.760,30.0
3,4,SO - 0005614,2021-01-01,Colgate-Pa Group,Export,Product 7,7,2338.3,16368.1,1028.852,Bloomington,IN,Indiana,Midwest,39.16533,-86.52639,2021,Q1,January,1,7201.964,9166.136,56.0
4,5,SO - 0005781,2021-01-01,Deseret Group,Wholesale,Product 8,8,2291.4,18331.2,1260.270,Manchester,CT,Connecticut,Northeast,41.77524,-72.52443,2021,Q1,January,1,10082.160,8249.040,45.0


In [99]:
df.columns.tolist()

['transaction_id',
 'order_number',
 'order_date',
 'customer_name',
 'channel',
 'product_name',
 'quantity',
 'unit_price',
 'revenue',
 'unit_cost',
 'city',
 'state',
 'state_name',
 'us_region',
 'lat',
 'lon',
 'order_year',
 'order_quarter',
 'order_month_name',
 'order_month_num',
 'total_cost',
 'profit',
 'profit_margin_percentage']

In [100]:
# Keep only the project analysis period: 2021–2024
analysis_df = df[
    df['order_year'].between(2021, 2024)
].copy()

print("Analysis dataset shape:", analysis_df.shape)
print("Years:", sorted(analysis_df['order_year'].unique()))

Analysis dataset shape: (61626, 23)
Years: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]


In [101]:
analysis_df.to_csv(
    'geoinsights_sales_2021_2024.csv',
    index=False
)

budget_df.to_csv(
    'product_budget_2024.csv',
    index=False
)